### Clasificación de Texto - Machine Learning

In [13]:
# Importar librerias
import pandas as pd

df = pd.read_csv("df_total.csv", sep=",")
df.head()

,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


In [14]:
# Viendo una noticia sin procesar
print(f"Noticia original: {df['news'][3]}")

Noticia original: Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual respecto al avance del 30 de marzo y se sitúa 22 puntos por encima del dato de febrero que ascendió al 76.De esos 22 puntos de diferencia la mayor parte la colocó el grupo de la vivienda 09 puntos por la subida de la electricidad y el del transporte 07 puntos por el alza de los carburantes. También impulsaron el IPC de marzo el aumento de los precios de la restauración y los servicios de alojamiento y al encarecimiento generalizado de los alimentos especialmente del pescado y el marisco de la carne de las legumbres y hortalizas y de la leche el queso y los huevos.Sin tener en cuenta la rebaja del impuesto especial sobre la electricidad y las variaciones sobre otros impuestos el IPC interanual alcanzó en marzo 107 nueve décimas más que la tasa general del 98. Así lo refleja el IPC a impuestos constantes que el INE también pu

#### Preparación y división de datos

In [15]:
from sklearn.model_selection import train_test_split

X = df['news']
y = df['Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Vectorización

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

#Vemos que X_train_transformed te mantiene las filas de datos, pero tiene un vector de n columnas
X_train_transformed_dense = X_train_transformed.toarray()
print(X_train_transformed_dense)

[[0 2 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 2 0 ... 0 0 0]]


#### Creación del modelo y del entrenamiento

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

# Algoritmo basado en probabilidad, excelente para texto
model = MultinomialNB() 

# Entrenando el modelo
model.fit(X_train_transformed, y_train)

# El modelo predice para adivinar las etiquetas del test
y_pred = model.predict(X_test_transformed) 

# Comparamos aciertos vs realidad
print(metrics.accuracy_score(y_test, y_pred)) 

0.7991803278688525


#### Mejora con Stemming

In [18]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

nltk.download('punkt')
stemmer = SnowballStemmer('spanish')

# Función para procesar el texto
def tokenize_and_stem(text):
    tokens = word_tokenize(text.lower())

# Solo nos quedamos con letras (quitamos signos de puntuación) y aplicamos stemmer
    stems = [stemmer.stem(token) for token in tokens if token.isalpha()]
    return ' '.join(stems)

# Aplicamos la función a todo el dataset
df['news_stemmer'] = df['news'].apply(tokenize_and_stem)

# REPETIMOS EL PROCESO CON LOS NUEVOS DATOS
X = df['news_stemmer']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# IMPORTANTE: Hay que volver a ajustar el vectorizador a las nuevas palabras (raíces)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(f"Precisión con Stemming: {metrics.accuracy_score(y_test, y_pred)}")

# El modelo ha mejorado porque el vocabulario es más pequeño y eficiente porque agrupamos palabras similares bajo una misma raíz

[nltk_data] Downloading package punkt to /home/ciabd10/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Precisión con Stemming: 0.8278688524590164


#### Mejora con Lemmatización (Diccionario)

In [19]:
import spacy
# Cargamos el modelo en español de spacy
nlp = spacy.load('es_core_news_sm')

def lemmatize_text(text):
    doc = nlp(text.lower())

    # Extraemos el lema de cada palabra si es una letra
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    return ' '.join(lemmas)

# Aplicamos lematización (esto puede tardar unos minutos)
df['news_lemma'] = df['news'].apply(lemmatize_text)

# REPETIMOS EL PROCESO CON LOS DATOS LEMATIZADOS
X = df['news_lemma']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Volvemos a transformar los datos
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(f"Precisión con Lematización: {metrics.accuracy_score(y_test, y_pred)}")

# Usamos spacy para entender gramaticalmente el texto. Es más preciso que el Stemming 
# porque no "corta" palabras al azar, sino que usa reglas del lenguaje español.

Precisión con Lematización: 0.8319672131147541
